# 🍽️ Yelp Review Summarizer - Colab Runner

This notebook sets up and runs the NLP Final Project on Google Colab with GPU acceleration.

**Before running:** Make sure to select a GPU runtime:
- Go to `Runtime` → `Change runtime type` → Select `L4 GPU`

---

## 1️⃣ Mount Google Drive (for persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2️⃣ Clone or Update the Repository

In [ ]:
import os

REPO_URL = "https://github.com/chucey/NLP-Final--Project.git"
REPO_DIR = "/content/NLP-Final--Project"

if os.path.exists(REPO_DIR):
    print("📦 Repo already exists, pulling latest changes...")
    !cd {REPO_DIR} && git pull
else:
    print("📥 Cloning repository...")
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"\n✅ Working directory: {os.getcwd()}")
!ls -la

## 3️⃣ Install Dependencies

⚠️ We only install project-specific packages here. Colab already has `torch`, `numpy`, `pandas`, etc. pre-installed — using `requirements.txt` directly would cause version conflicts.

In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    sentence-transformers \
    python-dotenv \
    accelerate

## 4️⃣ Verify GPU Availability

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU device:      {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:      {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print("\n🚀 GPU is ready! Everything will run much faster.")
else:
    print("\n⚠️ No GPU detected. Go to Runtime → Change runtime type → Select L4 GPU")

## 5️⃣ Build FAISS Index

This step generates embeddings for all reviews and builds the vector index.  
- **First run:** Takes a few minutes with GPU (vs 10-20 min on CPU)
- **After first run:** You can save the index to Google Drive and skip this step next time

In [ ]:
import os

DRIVE_INDEX_DIR = "/content/drive/MyDrive/NLP-Project/faiss_yelp"
LOCAL_INDEX_DIR = "faiss_yelp"

# Try to load from Google Drive first (skip rebuilding)
if os.path.exists(DRIVE_INDEX_DIR) and os.listdir(DRIVE_INDEX_DIR):
    print("📂 Found FAISS index on Google Drive, copying to local...")
    !cp -r {DRIVE_INDEX_DIR} {LOCAL_INDEX_DIR}
    print("✅ Index loaded from Drive! Skipping rebuild.")
else:
    print("🔨 Building FAISS index from scratch (this may take a few minutes)...")
    from build_rag import build_index
    build_index("data/all_reviews_dataset.csv")
    
    # Save to Google Drive for future sessions
    print("\n💾 Saving index to Google Drive for next time...")
    os.makedirs(os.path.dirname(DRIVE_INDEX_DIR), exist_ok=True)
    !cp -r {LOCAL_INDEX_DIR} {DRIVE_INDEX_DIR}
    print("✅ Index saved to Drive!")

## 6️⃣ Test Retrieval

Quick test to make sure the FAISS index works properly.

In [ ]:
import rag_retrival

print("Loading vectorstore...")
vs = rag_retrival.load_vectorstore()
print("✅ Vectorstore loaded!\n")

# Test retrieval with a sample query
results = rag_retrival.retrieve_reviews_for_summary(vs, categories="Italian", k=3)
print("🔍 Sample retrieval (Italian restaurants, top 3):")
print(results)

## 7️⃣ Run Full Summarization Pipeline

Load the LLM and generate a review summary. Modify the parameters below to summarize different businesses/categories.

In [ ]:
# ============================================================
# 👇 Modify these parameters to query different businesses
# ============================================================
CATEGORIES = "Italian"       # e.g., "Italian", "Mexican", "Coffee"
BUSINESS_NAME = None         # e.g., "Olive Garden" or None for all
CITY = None                  # e.g., "Philadelphia" or None for all
STATE = None                 # e.g., "PA" or None for all
REVIEW_STARS = None          # e.g., 5 or None for all
K = 80                       # Number of reviews to retrieve
# ============================================================

In [ ]:
import rag_retrival
from prompt import load_model, summarize_reviews

MODEL_NAME = "Qwen/Qwen3-0.6B"

# Step 1: Load vectorstore
print("=" * 50)
print("📚 Loading vectorstore...")
vs = rag_retrival.load_vectorstore()
print("✅ Vectorstore loaded.")

# Step 2: Retrieve reviews
print("\n" + "=" * 50)
print("🔍 Retrieving reviews...")
docs = rag_retrival.retrieve_reviews_for_summary(
    vs,
    business_name=BUSINESS_NAME,
    city=CITY,
    state=STATE,
    review_stars=REVIEW_STARS,
    categories=CATEGORIES,
    k=K
)
print(f"✅ Retrieved reviews.")

# Step 3: Load LLM
print("\n" + "=" * 50)
print("🤖 Loading language model...")
tok, model = load_model(MODEL_NAME)
print("✅ Model loaded.")

# Step 4: Generate summary
print("\n" + "=" * 50)
print("✍️ Generating summary...")
summary = summarize_reviews(docs, tok, model)

print("\n" + "=" * 50)
print("📝 BUSINESS REVIEW SUMMARY")
print("=" * 50)
print(summary)

---

## 🔧 Utility: Update Repo from Teammates

Run this cell anytime your teammates push changes:

In [ ]:
!cd /content/NLP-Final--Project && git pull
print("\n✅ Repository updated!")